# Can a least-RMS fit in the time domain recover the coefficients with which known inputs enter an output?

When an output contains scaled copies of measured inputs plus independent noise, the scale factors can be estimated by choosing the linear combination of inputs that minimizes the RMS of the corrected output. The notebook tests this on synthetic data whose coefficients are known, first with one input and then with three. The estimate can fail when the inputs are not independent over the finite record, so the fit can trade one coefficient against another, and when the minimizer stops short of the least-squares solution.

The `optimal_linear_combination` docstring states the target

$$\min_{c}\ \mathrm{RMS}\Big[\,y + \sum_i c_i\,x_i\Big],$$

evaluated here without timeshifts and with each series' mean removed first. Because the inputs are added, the recovered $c_i$ is the negative of the coefficient with which $x_i$ enters $y$.

The single-input data are $y = -2.5\,x + n$ with $x$ a 1 Hz sinusoid. The three-input data are $y = 1.5\,x_1 - 0.8\,x_2 + 2.0\,x_3 + n$, with $x_1$ a 1 Hz sine, $x_2$ a 0.5 Hz cosine, and $x_3 = e^{-0.1t}$. Each record has 1000 samples over 10 s, and $n$ is white with standard deviation 0.1. `speckit.dsp.optimal_linear_combination(df, inputs, output, domain="time")` minimizes the time-domain RMS with SciPy's TNC method at tolerance $10^{-9}$ and returns the optimizer result and the corrected output. No spectral density is computed, so the one-sided convention does not enter. The reference is the set of injected coefficients.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from speckit.dsp import optimal_linear_combination
from speckit.plotting import default_rc

plt.rcParams.update(default_rc)

In [ ]:
# Utility: create synthetic signals
def create_synthetic_data_1d(n=1000, noise_level=0.1, seed=0):
    np.random.seed(seed)
    t = np.linspace(0, 10, n)
    x = np.sin(2 * np.pi * 1.0 * t)  # input
    noise = noise_level * np.random.randn(n)
    y = -2.5 * x + noise  # output = -2.5 * x + noise
    return pd.DataFrame({"x": x, "y": y})


def create_synthetic_data_miso(n=1000, noise_level=0.1, seed=0):
    np.random.seed(seed)
    t = np.linspace(0, 10, n)
    x1 = np.sin(2 * np.pi * 1.0 * t)
    x2 = np.cos(2 * np.pi * 0.5 * t)
    x3 = np.exp(-0.1 * t)
    noise = noise_level * np.random.randn(n)
    y = 1.5 * x1 - 0.8 * x2 + 2.0 * x3 + noise
    return pd.DataFrame({"x1": x1, "x2": x2, "x3": x3, "y": y})

In [ ]:
# Run 1D test
df1d = create_synthetic_data_1d()
res_1d, residual_1d = optimal_linear_combination(
    df1d, inputs=["x"], output="y", domain="time"
)

print("\nRecovered coefficient:", res_1d.x)
plt.figure()
plt.title("Single-input single-output problem")
plt.plot(df1d["x"])
plt.plot(df1d["y"])
plt.plot(residual_1d)
plt.xlabel("Sample")
plt.ylabel("Residual")
plt.show()

In [ ]:
# Run MISO test
df_miso = create_synthetic_data_miso()
res_miso, residual_miso = optimal_linear_combination(
    df_miso, inputs=["x1", "x2", "x3"], output="y", domain="time"
)

print("\nRecovered coefficients:", res_miso.x)
plt.figure()
plt.title("Multiple-input single-output problem")
plt.plot(df_miso["x1"])
plt.plot(df_miso["x2"])
plt.plot(df_miso["x3"])
plt.plot(df_miso["y"])
plt.plot(residual_miso)
plt.xlabel("Sample")
plt.ylabel("Residual")
plt.show()

## Reading the result

Each test prints the recovered coefficients and plots the inputs, the output, and the corrected output on one axis. The comparison that decides the notebook is the printed `res.x` against the negated injected coefficients: 2.5 for the single-input case, and −1.5, 0.8, and −2.0 for $x_1$, $x_2$, and $x_3$. In each plot the corrected output should look like white noise with a standard deviation near the injected 0.1, with no visible trace of the sinusoids or the exponential.

Limits in the code: every series has its mean removed before fitting, so the corrected output is centered on zero. Over a 10 s record the three inputs are not orthogonal after mean removal; that does not bias the fit, but it couples the errors of the three coefficients. The result is the minimizer's solution at tolerance $10^{-9}$, not a closed-form least-squares solve. `res.success` and `res.message` report whether TNC converged, and the function also logs them at INFO level. The y-axis label reads "Residual" for all traces.